[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/09_formats_complexity.ipynb)

# Formats complexity — comparing modes of convening

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-18

**Abstract**: Public-dialogue studies often ask whether *how* people are convened changes
*what* they raise — workshop versus survey, chatbot versus in-person. Multilogue
does not hard-code “format”; it supplies the Address machinery with a group
column and a consumer metric $\ell$. This notebook teaches that pattern: Access
on a public format pair; a paper-aligned gold $\ell$ (major position vs premise
by format); **Assess** that locks subject grain before spend; then Address that
prefers a cached CIMT LLM run at locked $k{=}10$ with German municipal cycling
lenses (resuming via StageRunner when an API key is present). Offline Address
uses `phrases_from_units` on the same CIMT units (FakeClient only when no key /
``MULTILOGUE_FORCE_FAKE``). Multiverse holds $k$ (or $\pm 1$) by default — not a
wide resolution grid.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

An `ai-dialogue-formats`-style study keeps PDF parsers, prompts, and the precise
definition of $\ell$ in the consumer package. Multilogue owns extract → cluster →
weight → soft lenses → document distance → multiverse. The group column here is
`format` $\in\{\mathrm{dialogue},\mathrm{survey}\}$. The headline $\ell$ is a
document-weighted format gap on the lens with largest absolute difference —
recomputed under each analytic configuration without new LLM calls.

The flowchart below is the map for this notebook. Section headings use the same
numbers and titles so you can jump from the diagram to the matching cell.


## Analysis flow

Eight numbered stages. **Assess** covers text diagnostics (CIP-0008) and locks
subject grain (Bishop–Tipping dual view) before **Address** spends on lenses /
$\ell$. Address expands into the soft-throughout pipeline that `StageRunner`
resumes from disk (extract → embed → MPPCA topics → labels/lenses → soft compare
/ EMD → multiverse $\ell$ with $k$ **held** or $\pm 1$ only).

```mermaid
flowchart TD
  S["1. Setup"] --> A["2. Access — public format pair"]
  A --> T["3. Assess — text diagnostics"]
  T --> G["4. Gold ℓ — mpos vs premise by format"]
  G --> AG["5. Assess — interactive subject grain"]
  AG --> AD["6. Address — soft-throughout pipeline"]

  subgraph ADDR["Address internals (one StageRunner run)"]
    direction TB
    E1["6a. Extract phrases"] --> E2["6b. Embed"]
    E2 --> E3["6c. Soft cluster — MPPCA + topic σ² profile"]
    E3 --> E4["6d. Topic labels + framing lenses"]
    E4 --> E5["6e. Soft compare, document EMD, multiverse ℓ (k held)"]
  end

  AD --> ADDR
  ADDR --> R["7. Reading the run"]
  R --> O["8. Outlook"]

  classDef stage fill:#e8f1fa,stroke:#2c5282,stroke-width:1.5px,color:#1a202c
  classDef inner fill:#f7fafc,stroke:#718096,stroke-width:1px,color:#1a202c
  class S,A,T,G,AG,AD,R,O stage
  class E1,E2,E3,E4,E5 inner
```

| # | Section | What it does |
|---|---------|--------------|
| 1 | Setup | Install / locate multilogue (first code cell) |
| 2 | Access — public format pair | CIMT dialogue vs survey |
| 3 | Assess — text diagnostics | Lengths / characters (CIP-0008); stop if debris dominates |
| 4 | Gold $\ell$ — mpos vs premise by format | Paper-aligned argument codes; no LLM |
| 5 | Assess — interactive subject grain | Bishop–Tipping dual view; lock $k$ before Address |
| 6 | Address — soft-throughout pipeline | Cache / live / phrases_from_units; StageRunner stages |
| 7 | Reading the run | Catalogue, top concerns, findings markdown |
| 8 | Outlook | What stays in a consumer formats package |


In [ ]:
# Drawn copy of the analysis flow (renders in Colab / plain Jupyter; Mermaid above is for GitHub).
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

stages = [
    (0.50, 0.96, "1. Setup"),
    (0.50, 0.88, "2. Access — public format pair"),
    (0.50, 0.80, "3. Assess — text diagnostics"),
    (0.50, 0.72, "4. Gold ℓ — mpos vs premise by format"),
    (0.50, 0.64, "5. Assess — interactive subject grain"),
    (0.50, 0.56, "6. Address — soft-throughout pipeline"),
]
inners = [
    (0.50, 0.46, "6a. Extract phrases"),
    (0.50, 0.38, "6b. Embed"),
    (0.50, 0.30, "6c. Soft cluster — MPPCA + topic σ² profile"),
    (0.50, 0.22, "6d. Topic labels + framing lenses"),
    (0.50, 0.14, "6e. Soft compare, EMD, multiverse ℓ (k held)"),
]
tail = [
    (0.50, 0.06, "7. Reading the run"),
    (0.50, -0.02, "8. Outlook"),
]

fig, ax = plt.subplots(figsize=(9.5, 12.0))
ax.set_xlim(0, 1)
ax.set_ylim(-0.06, 1.02)
ax.axis("off")
ax.set_title("Formats complexity — analysis flow", fontsize=14, pad=12)

def _box(x, y, text, *, face, edge, width=0.72, height=0.055, fontsize=10, weight="normal"):
    box = FancyBboxPatch(
        (x - width / 2, y - height / 2),
        width, height,
        boxstyle="round,pad=0.012,rounding_size=0.02",
        facecolor=face, edgecolor=edge, linewidth=1.4, transform=ax.transAxes,
        clip_on=False,
    )
    ax.add_patch(box)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize,
            fontweight=weight, transform=ax.transAxes, color="#1a202c")

def _arrow(y0, y1, gap=0.028):
    ax.annotate(
        "", xy=(0.50, y1 + gap), xytext=(0.50, y0 - gap),
        xycoords=ax.transAxes, textcoords=ax.transAxes,
        arrowprops=dict(arrowstyle="->", color="#2c5282", lw=1.3),
    )

frame = FancyBboxPatch(
    (0.08, 0.105), 0.84, 0.40,
    boxstyle="round,pad=0.01,rounding_size=0.02",
    facecolor="#f7fafc", edgecolor="#a0aec0", linewidth=1.0,
    linestyle="--", transform=ax.transAxes, clip_on=False,
)
ax.add_patch(frame)
ax.text(0.10, 0.490, "Address internals (one StageRunner run)",
        fontsize=9, color="#4a5568", transform=ax.transAxes, style="italic")

for x, y, t in stages:
    _box(x, y, t, face="#e8f1fa", edge="#2c5282", weight="bold")
for x, y, t in inners:
    _box(x, y, t, face="#edf2f7", edge="#718096", width=0.78, height=0.05, fontsize=9)
for x, y, t in tail:
    _box(x, y, t, face="#e8f1fa", edge="#2c5282", weight="bold")

chain = [s[1] for s in stages] + [i[1] for i in inners] + [t[1] for t in tail]
for a, b in zip(chain, chain[1:]):
    _arrow(a, b)

fig.tight_layout()
plt.show()
path = FIG / "09_analysis_flow.png"
fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
print("Saved", path)


In [ ]:
# Optional transport dependency (soft lenses + document EMD).
import importlib.util, subprocess, sys
if importlib.util.find_spec("ot") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pot"])
print("POT available")


## 2. Access — public format pair

CIMT Bonn cycling *dialogue* vs *survey* stand in for workshop vs survey.
Network is required once to populate the cache (`MULTILOGUE_DATAPATH` or
`~/.cache/multilogue`).


In [ ]:
import os
from IPython.display import display
import pandas as pd
from multilogue.experiment import load_cimt_format_pair
from multilogue.extract import phrases_from_units
from multilogue.assess_base import assess_text_report, docs_per_group
from multilogue import viz


In [ ]:
os.environ.setdefault("MULTILOGUE_ACCESS_YES", "1")
cimt_units, cimt_meta = load_cimt_format_pair(max_units_per=60)
if cimt_units is None:
    raise RuntimeError(
        "CIMT Access failed: "
        + str(cimt_meta.get("error", "unknown"))
        + ". Need network once to populate MULTILOGUE_DATAPATH / ~/.cache/multilogue."
    )
access_units = cimt_units
access_label = "CIMT Cycling-Dialogue-Bonn vs Survey-Bonn"
access_lang = "de"
display(cimt_meta)

print(access_label, "—", len(access_units), "units")
display(docs_per_group(access_units, "format"))
offline_phrases = phrases_from_units(access_units, extra_fields=["group", "format"])
print(f"{len(offline_phrases)} offline phrases (unit-as-phrase)")


## 3. Assess — text diagnostics

Question-agnostic checks (same helpers as notebook 01): are paragraphs a
sensible length? Do sentences and letter-tokens look sane for German? Is the
character inventory mostly letters/punctuation?

**Do not proceed to Address if** empty/too-short units dominate, sentence or
word lengths are wildly extreme, or ``other`` character share is high — fix
Access first (CIP-0008).


In [ ]:
report = assess_text_report(access_units, lang=access_lang)
print(
    f"empty={len(report['empty'])}; "
    f"unit_flags={len(report['unit_length_flags'])}; "
    f"sent_flags={len(report['sentence_flags'])}; "
    f"word_flags={len(report['word_flags'])}; "
    f"char_flags={len(report['char_flags'])}"
)
display(report["size_summary"])
viz.unit_size_hist(
    report["sizes"], group_col="format", units=access_units,
    path=FIG / "09_unit_sizes.png",
)
viz.sentence_length_hist(report["sentences"], path=FIG / "09_sentence_lengths.png")
viz.word_length_hist(report["words"], path=FIG / "09_word_lengths.png")
viz.char_class_bars(report["char_classes"], path=FIG / "09_char_classes.png")
display(report["char_classes"])
display(report["unusual_codepoints"].head(8))
display(report["unit_length_flags"].head())
display(report["char_flags"].head())


## 4. Gold $\ell$ — mpos vs premise by format

Romberg & Conrad (2021) report that map *dialogue* contributions are longer and
**premise-rich**, while the Bonn *survey* is short and **major-position-heavy**.
That contrast uses the dataset's gold `meta.code` (`mpos` / `premise`), not LLM
topics. The next cell is the paper-aligned headline $\ell$: document-weighted
shares of each code by format. It needs no API key.


In [ ]:
import importlib
import multilogue.experiment as _ml_exp
importlib.reload(_ml_exp)  # pick up edits without a full kernel restart
gold_argument_format_compare = _ml_exp.gold_argument_format_compare

gold_cmp = gold_argument_format_compare(access_units, formats=("dialogue", "survey"))
if len(gold_cmp):
    display(gold_cmp)
    for r in gold_cmp.itertuples():
        print(
            f"{r.metric}: dialogue {r.dialogue_pct:.1f}% vs survey {r.survey_pct:.1f}% "
            f"(Δ {r.difference:+.1f} pp)"
        )
else:
    print("No gold argument codes on these units.")


The sign of $\Delta$ (survey − dialogue) should be **positive for mpos** and
**negative for premise** if this slice reproduces the paper. Topic clustering
below is a separate question — first **Assess** the subject grain, then Address.


## 5. Assess — interactive subject grain

Subject count $k$ defines how many topics the analysis admits. Putting a wide
$k$ grid in the factorial multiverse swamps quote / kind / membership and
answers the wrong stability question. CIP-0009 treats grain as **Assess EDA
with a human in the loop** (Bishop–Tipping geometry + LLM catalogues); only
after a **lock** does Address proceed.

Notebook 03 teaches the dual-view shell on CIMT phrases. Here we record
the lock policy for this formats run and keep the offline path
scripted (no widgets required). **Lock grain before Address spend** — do not
re-fit $k$ to maximise format-gap $\ell$.


In [ ]:
import os
from pathlib import Path
from multilogue.subject_grain import lock_subject_grain

# Formats Address default: hold locked primary k (CIP-0009). Optional pm1
# via resolve_k_multiverse(10, "pm1") → [9, 10, 11]. Wide grids only via
# explicit k_grid= on run_formats_complexity_demo (one-factor sweep).
PRIMARY_K = 10
grain_lock = lock_subject_grain(
    primary_k=PRIMARY_K,
    k_multiverse="hold",
    extra={
        "note": "Assess lock before Address; dual-view merge/split lives in notebook 03",
        "lock_before_spend": True,
    },
)
lock_dir = Path("outputs/notebook_formats_complexity")
lock_dir.mkdir(parents=True, exist_ok=True)
from multilogue.config import dump_json
dump_json(grain_lock, lock_dir / "subject_grain_lock.json")
print("subject-grain lock →", lock_dir / "subject_grain_lock.json")
print("primary_k =", grain_lock["primary_k"], "k_multiverse =", grain_lock["k_multiverse"])
display(grain_lock)


## 6. Address — soft-throughout pipeline

StageRunner checkpoints under `outputs/formats_cimt_llm/` mean a prior CIMT LLM
run can be reused without paying again. Internally this is stages **5a–5e** on
the flowchart (extract → embed → MPPCA + topic $\sigma^2$ profile → labels/lenses
→ soft compare / EMD / multiverse $\ell$ with **$k$ held**). The next cell
therefore:

1. loads `.env` if present;
2. if `OPENAI_API_KEY` is set — *resume or recache* the live
   demo at the **locked** $k{=}10$ with **German labels** and a fixed municipal
   cycling lens vocabulary (paths, parking, surface, junctions/tram,
   access/Rhein, mixed traffic). Completed extract/embed fingerprints are
   reused; new prompts re-run cluster labels and lenses — do not sweep a wide
   $k$ grid here;
3. else if a complete cached run is already on disk — use those tables;
4. else run offline Address on the same CIMT units via `phrases_from_units`
   (OpenAI from ``.env`` when available; FakeClient offline — no synthetic civic text).


In [ ]:
import json
import os
import importlib
from pathlib import Path
from IPython.display import display, Markdown
from multilogue.config import load_dotenv
import multilogue.experiment as _ml_exp
importlib.reload(_ml_exp)
run_formats_complexity_demo = _ml_exp.run_formats_complexity_demo
CYCLING_LABEL_PROMPT = _ml_exp.CYCLING_LABEL_PROMPT
CYCLING_LENS_PROMPT = _ml_exp.CYCLING_LENS_PROMPT
from multilogue import viz
from multilogue.report import (
    cluster_catalogue, findings_markdown, top_phrases_by_group, write_findings,
)


In [ ]:
load_dotenv(Path.cwd() / ".env")
load_dotenv(Path.cwd().parent / ".env")
# Prefer repo-root outputs when the notebook cwd is notebooks/.
for _root in (Path.cwd(), Path.cwd().parent):
    _cand = _root / "outputs" / "formats_cimt_llm"
    if (_cand / "tables" / "phrases_clustered.csv").is_file() or (_cand / "checkpoints").is_dir():
        LLM_OUT = _cand
        break
else:
    LLM_OUT = Path("outputs/formats_cimt_llm")

CACHE_TABLES = LLM_OUT / "tables"
cache_ready = (CACHE_TABLES / "phrases_clustered.csv").is_file()
key = os.environ.get("OPENAI_API_KEY", "").strip()
summary = None
address_source = None

if key:
    from multilogue.openai_client import OpenAIClient
    units_live = [
        u for u in access_units
        if (u.get("meta") or {}).get("code") != "non-arg"
    ]
    print(
        f"Live / resume CIMT Address → {LLM_OUT} "
        f"(locked k={PRIMARY_K}, German cycling lenses; {len(units_live)} units)"
    )
    summary = run_formats_complexity_demo(
        LLM_OUT,
        units=units_live,
        client=OpenAIClient(log_path=LLM_OUT / "logs" / "openai.jsonl"),
        kinds=("concern", "benefit"),
        extract_model="gpt-4.1-mini",
        embed_model="text-embedding-3-small",
        label_model="gpt-4.1-mini",
        k=PRIMARY_K, lens_runs=3, quiet=True, max_workers=4,
        label_prompt_template=CYCLING_LABEL_PROMPT,
        lens_prompt_template=CYCLING_LENS_PROMPT,
        k_multiverse="hold",
        subject_grain_lock=grain_lock,
    )
    tables = CACHE_TABLES
    address_source = "cimt_llm"
elif cache_ready:
    print(f"Using cached CIMT Address tables → {CACHE_TABLES.resolve()}")
    tables = CACHE_TABLES
    address_source = "cimt_cache"
    man = LLM_OUT / "run_manifest.json"
    if man.is_file():
        summary = json.loads(man.read_text())
        display(summary)
        if summary.get("k", 0) < 10:
            print(
                "Note: cache used k =", summary.get("k"),
                "— re-run with OPENAI_API_KEY to refresh at k=10 + German lenses.",
            )
else:
    out = Path("outputs/notebook_formats_complexity")
    print(
        "No CIMT cache and no API key — offline phrases_from_units →", out
    )
    summary = run_formats_complexity_demo(
        out,
        units=access_units,
        phrases=offline_phrases,
        k=PRIMARY_K,
        lens_runs=3,
        quiet=True,
        k_multiverse="hold",
        subject_grain_lock=grain_lock,
    )
    tables = out / "tables"
    address_source = "cimt_offline"

display(summary)
print("Address source:", address_source)
print("Subject grain:", (summary or {}).get("subject_grain") if isinstance(summary, dict) else None)
print("Multiverse dims (no wide k by default):", (summary or {}).get("multiverse_dims") if isinstance(summary, dict) else None)
print("Tables:", tables.resolve())
print("Checkpoints:", LLM_OUT / "checkpoints" if address_source != "cimt_offline" else tables.parent / "checkpoints")


In [ ]:
for name in [
    "yield.csv",
    "gold_argument_format_compare.csv",
    "group_compare_format_lenses.csv",
    "soft_group_compare_format.csv",
    "sigma2_rules.csv",
    "doc_distance_pairwise.csv",
    "variance_shares_format_gap.csv",
]:
    p = tables / name
    if p.is_file():
        print("—", name)
        display(pd.read_csv(p).head(12))

shares_path = tables / "variance_shares_format_gap.csv"
if shares_path.is_file():
    shares = pd.read_csv(shares_path)
    viz.variance_share_bars(shares, path=FIG / "09_format_gap_variance.png")
    # CIP-0009: factorial dims should not be dominated by a silent k grid
    if "dimension" in shares.columns and (shares["dimension"] == "k").any():
        print("Note: k appears in variance shares — pass k_grid= only for deliberate sweeps.")
if isinstance(summary, dict):
    print("Soft:", summary.get("soft"))
    print("Distance:", summary.get("distance"))
    print("Locked subject grain:", summary.get("subject_grain"))


## 7. Reading the run

Findings summarise gold argument structure (when present), clusters, top
concerns by format, and which analytic decisions move the topic-gap $\ell$.
Multiverse variance shares here are over quote / kind / membership — **not** a
wide $k$ grid (stage **5e** holds locked grain). This cell only reads tables —
no LLM calls.


In [ ]:
ph = pd.read_csv(tables / "phrases_clustered.csv")
cmp = (
    pd.read_csv(tables / "group_compare_format_lenses.csv")
    if (tables / "group_compare_format_lenses.csv").is_file() else None
)
multi = (
    pd.read_csv(tables / "multiverse_format_gap.csv")
    if (tables / "multiverse_format_gap.csv").is_file() else None
)
shares = (
    pd.read_csv(tables / "variance_shares_format_gap.csv")
    if (tables / "variance_shares_format_gap.csv").is_file() else None
)
gold_path = tables / "gold_argument_format_compare.csv"
gold_tab = pd.read_csv(gold_path) if gold_path.is_file() else (
    gold_cmp if len(gold_cmp) else None
)
display(Markdown("### Cluster catalogue"))
display(cluster_catalogue(ph))
display(Markdown("### Top concerns by format"))
for g, tab in top_phrases_by_group(ph, kind="concern", n=8).items():
    display(Markdown(f"**{g}**"))
    display(tab)
if shares is not None:
    display(Markdown("### Multiverse variance shares"))
    display(shares)
title = (
    "CIMT dialogue vs survey — findings"
    if address_source in {"cimt_llm", "cimt_cache"}
    else "CIMT dialogue vs survey — findings (offline phrases)"
)
md_text = findings_markdown(
    ph, compare=cmp, multiverse=multi, variance_shares=shares,
    gold_compare=gold_tab, title=title,
)
write_findings(
    ph, tables / "findings.md", compare=cmp,
    multiverse=multi, variance_shares=shares,
    gold_compare=gold_tab, title=title,
)
display(Markdown(md_text))


## 8. Outlook

Multilogue owns the shared stages above. A formats-style consumer package still
keeps: PDF/docx Access, study prompts and experience coding, researcher-edited
lens YAML, and the precise definition of headline $\ell$. Subject grain is
locked in Assess (notebook 03 dual-view shell; this notebook's lock artefact)
before Address spend. See the formats cookbook and the soft / EMD concepts page
in the documentation.
